# Clinical Trial Designer — GRPO Training Notebook

**OpenEnv Hackathon | Meta PyTorch × Scaler School of Technology**

This notebook trains an LLM to design clinical trials using GRPO (Group Relative Policy Optimization) from HuggingFace TRL + Unsloth for efficient LoRA fine-tuning.

**Platform:** Google Colab (T4 GPU, 16 GB VRAM — free tier works with 4-bit quantization)

**What this notebook does:**
1. Installs dependencies (TRL, Unsloth)
2. Connects to the Clinical Trial Designer environment (HF Space)
3. Configures GRPO with LoRA
4. Trains the agent with reward feedback from the environment
5. Evaluates trained vs base model
6. Saves checkpoint to HuggingFace Hub

> **Note:** For dry-run validation (no GPU needed), set `DRY_RUN = True` in Cell 5.

## 1. Install Dependencies

In [ ]:
%%capture
# Install Unsloth for fast LoRA training (2x speed, 50% less VRAM)
!pip install unsloth
# Install TRL for GRPO trainer
!pip install trl>=0.29.0
# Install other dependencies
!pip install requests scipy numpy datasets

## 2. Configuration and Environment Connection

The environment runs on our HuggingFace Space. Change `ENV_URL` if running locally.

In [ ]:
import requests
import json
import os
import random
import re
import csv
from datetime import datetime, timezone

# === CONFIGURATION ===
ENV_URL = "https://roopalgn-openenv-clinical-trial.hf.space"
DRY_RUN = False          # Set True for pipeline validation without GPU/model
MODEL_SIZE = "1.5b"      # "1.5b", "3b", or "7b"
EPISODES = 20            # Training episodes
SEED = 42

# Model size presets (match train.py)
MODEL_PRESETS = {
    "1.5b": {"model_name": "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit", "lora_r": 8, "seq_len": 2048, "grad_accum": 4},
    "3b":   {"model_name": "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",   "lora_r": 16, "seq_len": 3072, "grad_accum": 4},
    "7b":   {"model_name": "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",   "lora_r": 16, "seq_len": 4096, "grad_accum": 8},
}

preset = MODEL_PRESETS[MODEL_SIZE]
print(f"Config: model={MODEL_SIZE}, episodes={EPISODES}, dry_run={DRY_RUN}")
print(f"Preset: {preset}")


def env_reset(seed=None):
    """Reset environment and get initial observation."""
    payload = {}
    if seed is not None:
        payload["seed"] = seed
    resp = requests.post(f"{ENV_URL}/reset", json=payload, timeout=30)
    resp.raise_for_status()
    return resp.json()

def env_step(action_type, parameters=None, justification="", confidence=0.5):
    """Take an action in the environment."""
    payload = {
        "action_type": action_type,
        "parameters": parameters or {},
        "justification": justification,
        "confidence": confidence,
    }
    resp = requests.post(f"{ENV_URL}/step", json=payload, timeout=30)
    resp.raise_for_status()
    return resp.json()

def env_ping():
    """Health check."""
    return requests.get(f"{ENV_URL}/ping", timeout=10).json()

# Test connection
try:
    print("Ping:", env_ping())
    obs = env_reset(seed=42)
    print("Reset OK. Scenario:", obs.get("scenario_description", "")[:80])
    print("Available actions:", obs.get("available_actions", []))
except Exception as e:
    print(f"ERROR: Cannot connect to environment at {ENV_URL}: {e}")
    print("Check that the HF Space is running.")

## 3. Dry-Run Validation (No GPU Needed)

Run 2 episodes with a random policy to validate the full pipeline: env connection, action, reward, CSV logging. Skip this section if doing real training.

In [ ]:
def run_dry_run(n_episodes=2, max_steps=50, base_seed=42):
    """Run N episodes with random policy, log rewards to CSV."""
    os.makedirs("outputs/grpo", exist_ok=True)
    csv_path = "outputs/grpo/reward_log.csv"
    results = []

    with open(csv_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["episode", "seed", "total_reward", "steps", "terminal_outcome", "timestamp"])

        for ep in range(n_episodes):
            ep_seed = base_seed + ep
            random.seed(ep_seed)
            obs = env_reset(seed=ep_seed)
            total_reward = 0.0
            steps = 0
            done = False

            while not done and steps < max_steps:
                actions = obs.get("available_actions", ["synthesize_conclusion"])
                action_type = random.choice(actions)
                result = env_step(action_type)
                reward = result.get("reward", {})
                step_reward = sum(reward.values()) if isinstance(reward, dict) else float(reward)
                total_reward += step_reward
                done = result.get("done", False)
                obs = result.get("observation", obs)
                steps += 1

            outcome = "success" if done else "timeout"
            ts = datetime.now(timezone.utc).isoformat()
            writer.writerow([ep, ep_seed, f"{total_reward:.6f}", steps, outcome, ts])
            results.append(total_reward)
            print(f"  Episode {ep+1}/{n_episodes} | reward={total_reward:.4f} | steps={steps} | {outcome}")

    print(f"\nDry-run complete. CSV: {csv_path}")
    print(f"Mean reward: {sum(results)/len(results):.4f}")
    return results

if DRY_RUN:
    print("=== DRY RUN MODE ===")
    run_dry_run(n_episodes=2, base_seed=SEED)
    print("=== Pipeline validated. Set DRY_RUN=False for real training. ===")

## 4. Load Model with Unsloth

4-bit quantization to fit on Colab T4 (16 GB VRAM). Model size is set by `MODEL_SIZE` in Cell 5.

In [ ]:
if not DRY_RUN:
    from unsloth import FastLanguageModel

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=preset["model_name"],
        max_seq_length=preset["seq_len"],
        dtype=None,
        load_in_4bit=True,
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r=preset["lora_r"],
        lora_alpha=preset["lora_r"] * 2,
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        bias="none",
        use_gradient_checkpointing="unsloth",
    )

    print(f"Model loaded: {preset['model_name']}")
    model.print_trainable_parameters()
else:
    print("DRY_RUN=True — skipping model load.")

## 4. Define Reward Function

The reward function runs a full episode in the environment for each completion. The agent's response is parsed as a JSON action, stepped through the environment, and the cumulative reward is returned.

**Reward range:** -3 to +14 (high variance for GRPO advantage computation).

In [ ]:
SYSTEM_PROMPT = """You are a clinical trial designer. You receive a disease scenario
and must design a complete Phase I/II clinical trial.

Your goal: design a trial that detects the drug's true effect and identifies
the right patient population, within budget and FDA regulations.

Available actions (respond with JSON):
PHASE I: run_dose_escalation, observe_safety_signal, estimate_effect_size
PHASE II: set_primary_endpoint, set_sample_size, set_inclusion_criteria,
    set_exclusion_criteria, set_dosing_schedule, set_control_arm,
    set_randomization_ratio, set_blinding
REGULATORY: submit_to_fda_review, request_protocol_amendment
MONITORING: run_interim_analysis, modify_sample_size, add_biomarker_stratification
ANALYSIS: run_primary_analysis
CONCLUSION: synthesize_conclusion

Follow Phase I -> Phase II -> FDA -> Monitoring -> Analysis -> Conclusion.
Respond: {"action_type": "<action>", "parameters": {...}}"""


def parse_action(text):
    """Extract JSON action from model output."""
    match = re.search(r'\{[^{}]*"action_type"[^{}]*\}', text)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass
    return {"action_type": "synthesize_conclusion", "parameters": {}}


def run_episode(model_response):
    """Run a full episode from a model's multi-step plan."""
    try:
        obs = env_reset(seed=random.randint(0, 10000))
        total_reward = 0.0
        done = False
        steps = 0
        max_steps = 50

        actions = []
        for line in model_response.split('\n'):
            parsed = parse_action(line)
            if parsed["action_type"] != "synthesize_conclusion" or not actions:
                actions.append(parsed)
        if not actions:
            actions = [parse_action(model_response)]

        for action in actions:
            if done or steps >= max_steps:
                break
            result = env_step(action["action_type"], action.get("parameters", {}))
            reward = result.get("reward", 0.0)
            if isinstance(reward, dict):
                reward = sum(reward.values())
            total_reward += reward
            done = result.get("done", False)
            steps += 1

        return total_reward
    except Exception as e:
        print(f"Episode error: {e}")
        return -2.0


def reward_func(completions, **kwargs):
    """GRPO reward function."""
    rewards = []
    for completion in completions:
        if isinstance(completion, list):
            text = completion[-1]["content"] if completion else ""
        else:
            text = str(completion)
        reward = run_episode(text)
        rewards.append(reward)
    return rewards

if not DRY_RUN:
    print("Reward function defined. Range: -3 to +14")

## 5. Prepare Training Dataset

GRPO needs prompts to generate completions for. Each prompt describes a clinical trial scenario. The model generates an action plan, which is scored by the environment.

In [ ]:
if not DRY_RUN:
    from datasets import Dataset

    SCENARIO_PROMPTS = [
        {
            "prompt": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": """Scenario: solid_tumor_chemo
Disease: Non-small cell lung cancer (NSCLC)
Drug: Novel EGFR tyrosine kinase inhibitor (3rd generation)
Budget: $2,500,000 | Time: 180 days
Challenge: Design a Phase I/II trial for this drug.

Plan your complete trial design step by step. For each step, provide a JSON action."""}
            ],
        },
        {
            "prompt": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": """Scenario: autoimmune_biologic
Disease: Rheumatoid arthritis (RA)
Drug: Novel anti-IL-6 biologic
Budget: $1,800,000 | Time: 150 days
Challenge: The drug may have a U-shaped dose-response. Design a trial to find the optimal dose.

Plan your complete trial design step by step. For each step, provide a JSON action."""}
            ],
        },
        {
            "prompt": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": """Scenario: cns_depression
Disease: Treatment-resistant depression (TRD)
Drug: Novel rapid-acting antidepressant
Budget: $3,000,000 | Time: 210 days
Challenge: High placebo response rate may mask the true drug effect. Design carefully.

Plan your complete trial design step by step. For each step, provide a JSON action."""}
            ],
        },
        {
            "prompt": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": """Scenario: rare_disease_orphan
Disease: Morquio A syndrome (rare pediatric metabolic disorder)
Drug: Enzyme replacement therapy
Budget: $1,200,000 | Time: 240 days
Challenge: Very small patient population (max ~50). Adaptive design required.

Plan your complete trial design step by step. For each step, provide a JSON action."""}
            ],
        },
    ]

    repeats = max(1, EPISODES // len(SCENARIO_PROMPTS))
    train_prompts = SCENARIO_PROMPTS * repeats
    train_dataset = Dataset.from_list(train_prompts)
    print(f"Training dataset: {len(train_dataset)} prompts ({len(SCENARIO_PROMPTS)} scenarios × {repeats} repeats)")
else:
    print("DRY_RUN=True — skipping dataset creation.")

## 6. Configure and Run GRPO Training

GRPO generates `num_generations` completions per prompt, scores them with the reward function, and updates the policy to favor higher-reward completions.

In [ ]:
if not DRY_RUN:
    from trl import GRPOConfig, GRPOTrainer

    training_args = GRPOConfig(
        output_dir="checkpoints/grpo_clinical_trial",
        num_generations=8,
        max_completion_length=512,
        temperature=0.7,
        learning_rate=5e-6,
        num_train_epochs=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=preset["grad_accum"],
        max_steps=EPISODES,
        warmup_ratio=0.05,
        weight_decay=0.01,
        max_grad_norm=1.0,
        logging_steps=1,
        save_steps=max(1, EPISODES // 4),
        report_to="none",
        bf16=True,
    )

    trainer = GRPOTrainer(
        model=model,
        args=training_args,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        reward_funcs=[reward_func],
    )

    print(f"GRPO Trainer configured. Steps: {EPISODES}, Rollouts/step: 8")
    print(f"Total episodes: ~{EPISODES * 8}")
else:
    print("DRY_RUN=True — skipping trainer setup.")

In [ ]:
if not DRY_RUN:
    print("Starting GRPO training...")
    print("=" * 60)
    trainer.train()
    print("=" * 60)
    print("Training complete!")
else:
    print("DRY_RUN=True — skipping training.")

## 7. Evaluate Trained Model

Compare the trained model against a random baseline to demonstrate learning.

In [ ]:
if not DRY_RUN:
    def random_policy(obs):
        actions = obs.get("available_actions", ["synthesize_conclusion"])
        return {"action_type": random.choice(actions), "parameters": {}}

    def trained_policy(obs):
        prompt = f"""Current observation: {json.dumps(obs, default=str)[:500]}
Select your next action as JSON: {{"action_type": "...", "parameters": {{...}}}}"""
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ]
        inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to(model.device)
        outputs = model.generate(inputs, max_new_tokens=256, temperature=0.3, do_sample=True)
        response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
        return parse_action(response)

    def evaluate_policy(policy_fn, n_episodes=10):
        results = []
        for i in range(n_episodes):
            obs = env_reset(seed=SEED + 1000 + i)
            total_reward = 0.0
            done = False
            steps = 0
            while not done and steps < 50:
                action = policy_fn(obs)
                result = env_step(action["action_type"], action.get("parameters", {}))
                reward = result.get("reward", 0.0)
                if isinstance(reward, dict):
                    reward = sum(reward.values())
                total_reward += reward
                done = result.get("done", False)
                obs = result.get("observation", obs)
                steps += 1
            results.append(total_reward)
        return results

    print("Evaluating random baseline (10 episodes)...")
    random_results = evaluate_policy(random_policy, n_episodes=10)
    print("Evaluating trained model (10 episodes)...")
    trained_results = evaluate_policy(trained_policy, n_episodes=10)

    random_avg = sum(random_results) / len(random_results)
    trained_avg = sum(trained_results) / len(trained_results)

    print(f"\n{'='*50}")
    print(f"{'Metric':<25} {'Random':>10} {'Trained':>10}")
    print(f"{'='*50}")
    print(f"{'Avg Reward':<25} {random_avg:>10.2f} {trained_avg:>10.2f}")
    print(f"{'Improvement':<25} {'':>10} {trained_avg - random_avg:>+10.2f}")
    print(f"{'='*50}")
else:
    print("DRY_RUN=True — skipping evaluation.")

## 8. Save Checkpoint to HuggingFace Hub

Upload the trained LoRA adapter to HuggingFace for deployment and demo.

In [ ]:
if not DRY_RUN:
    from huggingface_hub import login
    try:
        from google.colab import userdata
        login(token=userdata.get('HF_TOKEN'))
        print("Logged in to HuggingFace via Colab secret.")
    except Exception:
        print("Set HF_TOKEN in Colab Secrets (key icon in sidebar) or login manually:")
        from huggingface_hub import notebook_login
        notebook_login()
else:
    print("DRY_RUN=True — skipping HF login.")

In [ ]:
if not DRY_RUN:
    REPO_ID = "Roopalgn/clinical-trial-designer-grpo"

    model.save_pretrained("checkpoints/grpo_clinical_trial/final")
    tokenizer.save_pretrained("checkpoints/grpo_clinical_trial/final")

    model.push_to_hub(REPO_ID, commit_message=f"GRPO {MODEL_SIZE} trained on Colab")
    tokenizer.push_to_hub(REPO_ID, commit_message=f"GRPO {MODEL_SIZE} trained on Colab")

    print(f"Model uploaded to: https://huggingface.co/{REPO_ID}")
else:
    print("DRY_RUN=True — skipping checkpoint save.")

## 9. Generate Reward Curve Plot

Quick visualization of training progress for the pitch.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if not DRY_RUN and 'trainer' in dir():
    if hasattr(trainer, 'state') and trainer.state.log_history:
        rewards = [log.get('reward', None) for log in trainer.state.log_history if 'reward' in log]
        if rewards:
            episodes_list = range(1, len(rewards) + 1)
            window = min(20, len(rewards) // 3 + 1)
            rolling_avg = np.convolve(rewards, np.ones(window)/window, mode='valid')
            z = np.polyfit(range(len(rewards)), rewards, 1)
            trend = np.poly1d(z)

            fig, ax = plt.subplots(figsize=(12, 6))
            ax.scatter(episodes_list, rewards, alpha=0.3, color='#4a90d9', s=20, label='Per-episode')
            ax.plot(range(window, len(rewards) + 1), rolling_avg, color='#e63946',
                    linewidth=2, label=f'Rolling avg (w={window})')
            ax.plot(episodes_list, trend(range(len(rewards))), '--', color='#2d6a4f',
                    linewidth=1.5, label=f'Trend (slope={z[0]:.3f})')
            ax.set_xlabel('Episode', fontsize=12)
            ax.set_ylabel('Total Reward', fontsize=12)
            ax.set_title(f'Training Reward Curve — {MODEL_SIZE} on Colab', fontsize=14)
            ax.legend(loc='upper left')
            ax.grid(True, alpha=0.3)
            ax.annotate(
                f'Best: {max(rewards):.1f}\nFinal avg: {np.mean(rewards[-20:]):.1f}\nSlope: {z[0]:.3f}',
                xy=(0.98, 0.02), xycoords='axes fraction',
                ha='right', va='bottom', fontsize=10,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8)
            )
            os.makedirs("results", exist_ok=True)
            plt.tight_layout()
            plt.savefig('results/reward_curve.png', dpi=150)
            plt.show()
            print("Saved to results/reward_curve.png")
        else:
            print("No reward data in training logs.")
    else:
        print("No training logs available.")
else:
    print("DRY_RUN or no trainer — skipping plot.")

## Summary

This notebook demonstrates:
1. **Environment Innovation** — a clinical trial simulator with hidden ground truth, multi-layer verification, and 5-tier curriculum
2. **Reward Design** — 8 per-step + 7 terminal components with potential-based shaping
3. **Training** — GRPO with Unsloth/TRL producing measurable improvement
4. **Proof of Learning** — reward curves trending upward, trained model beating random baseline

For full documentation, see the [GitHub repository](https://github.com/Roopalgn/openenv-clinical-trial).